### Load Data and Preparation for Running


In [3]:
# Import Packages
import os
import kagglehub
import pandas as pd
import numpy as np

/Users/couci/UTokyo/Research/SyntheticCancerClassification/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Functions


In [2]:
def create_index(parent_folder: str):
    if "annotations.csv" in parent_folder:
        return
    index = []
    for root, dirs, files in os.walk(parent_folder):  # for each project
        for dir in dirs:  # for each sample
            if (
                dir.startswith(".") or dir == "annotations.csv"
            ):  # ignore hidden directories
                continue
            sample = {}
            sample["id"] = dir
            sample["project"] = root.split("/")[-1]
            dir = os.path.join(root, dir)
            sample["RNA-path"] = pd.NA
            sample["meth-path"] = pd.NA
            sample["mirna-path"] = pd.NA
            for file in os.listdir(dir):  # for each file
                if "RNA" in file:
                    sample["RNA-path"] = os.path.join(dir, file)
                if "methylation" in file:
                    sample["meth-path"] = os.path.join(dir, file)
                if "mirna" in file:
                    sample["mirna-path"] = os.path.join(dir, file)

            index.append(sample)

    index = pd.DataFrame(index)
    print("writing to ", "indices/" + parent_folder.split("/")[-1] + "_index.tsv")
    index = index.sort_values(by=["id"])
    index.to_csv(
        "indices/" + parent_folder.split("/")[-1] + "_index.tsv", sep="\t", index=False
    )

In [3]:
def merge_indices(parent_folder: str):
    indices = []
    for root, dirs, files in os.walk(parent_folder):
        for file in files:
            if "index.tsv" in file:
                indices.append(pd.read_csv(os.path.join(root, file), sep="\t"))

    index = pd.concat(indices, ignore_index=True, sort=True)
    print("writing to", os.path.join(parent_folder, f"root_index.tsv"))
    index.to_csv(os.path.join(parent_folder, f"root_index.tsv"), sep="\t", index=False)

#### Dataset Download


In [4]:
datasets_path = os.path.abspath("Datasets")
# Create Datasets folder if it doesn't exist
os.makedirs(datasets_path, exist_ok=True)
# Set KAGGLEHUB cache directory
os.environ["KAGGLEHUB_CACHE"] = datasets_path

# Download latest version to Datasets folder
download_path = kagglehub.dataset_download("ahmedhamdi906/tcga-pancan")

print("Path to dataset files:", download_path)

# Create indexs
os.makedirs("indices", exist_ok=True)

Path to dataset files: /Users/couci/UTokyo/Research/SyntheticCancerClassification/Datasets/datasets/ahmedhamdi906/tcga-pancan/versions/6


#### Dataset Setup


In [5]:
datasets_path = download_path
root = datasets_path
print("Root path:", root)
for path in os.listdir(root):
    create_index(f"{root}/{path}/{path}")

merge_indices("./indices/")

df = pd.read_csv("./indices/root_index.tsv", sep="\t")

annotations = pd.read_csv(datasets_path + "/annotations.csv")
annotations = annotations.dropna(subset=["OTHER_PATIENT_ID"])
annotations = annotations.drop(["SUBTYPE", "id", "study"], axis=1)
annotations["OTHER_PATIENT_ID"] = annotations["OTHER_PATIENT_ID"].str.lower()

merged_df = pd.merge(
    df, annotations, left_on="id", right_on="OTHER_PATIENT_ID", sort=True
)
merged_df.head()

Root path: /Users/couci/UTokyo/Research/SyntheticCancerClassification/Datasets/datasets/ahmedhamdi906/tcga-pancan/versions/6
writing to  indices/TCGA-LUSC_index.tsv
writing to  indices/TCGA-STAD_index.tsv
writing to  indices/TCGA-UVM_index.tsv
writing to  indices/TCGA-LAML_index.tsv
writing to  indices/TCGA-MESO_index.tsv
writing to  indices/TCGA-LIHC_index.tsv
writing to  indices/TCGA-KIRP_index.tsv
writing to  indices/TCGA-KIRC_index.tsv
writing to  indices/TCGA-LUAD_index.tsv
writing to  indices/TCGA-UCEC_index.tsv
writing to  indices/TCGA-BRCA_index.tsv
writing to  indices/TCGA-LGG_index.tsv
writing to  indices/TCGA-DLBC_index.tsv
writing to  indices/TCGA-TGCT_index.tsv
writing to  indices/TCGA-PRAD_index.tsv
writing to  indices/TCGA-ESCA_index.tsv
writing to  indices/TCGA-HNSC_index.tsv
writing to  indices/TCGA-KICH_index.tsv
writing to  indices/TCGA-GBM_index.tsv
writing to  indices/TCGA-OV_index.tsv
writing to  indices/TCGA-SKCM_index.tsv
writing to  indices/TCGA-PAAD_index.tsv


,RNA-path,id,meth-path,mirna-path,project,OTHER_PATIENT_ID,CANCER_TYPE,CANCER_TYPE_DETAILED,TUMOR_TISSUE_SITE,TUMOR_TYPE
0,/Users/couci/UTokyo/Research/SyntheticCancerCl...,0004d251-3f70-4395-b175-c94c2f5b1b81,/Users/couci/UTokyo/Research/SyntheticCancerCl...,/Users/couci/UTokyo/Research/SyntheticCancerCl...,TCGA-LIHC,0004d251-3f70-4395-b175-c94c2f5b1b81,Hepatobiliary Cancer,Hepatocellular Carcinoma,Liver,Hepatocellular Carcinoma
1,/Users/couci/UTokyo/Research/SyntheticCancerCl...,000d566c-96c7-4f1c-b36e-fa2222467983,/Users/couci/UTokyo/Research/SyntheticCancerCl...,/Users/couci/UTokyo/Research/SyntheticCancerCl...,TCGA-PRAD,000d566c-96c7-4f1c-b36e-fa2222467983,Prostate Cancer,Prostate Adenocarcinoma,Prostate,"Prostate Adenocarcinoma, Acinar Type"
2,/Users/couci/UTokyo/Research/SyntheticCancerCl...,001887aa-36d0-463f-8bca-dec7043b4f2e,/Users/couci/UTokyo/Research/SyntheticCancerCl...,/Users/couci/UTokyo/Research/SyntheticCancerCl...,TCGA-LIHC,001887aa-36d0-463f-8bca-dec7043b4f2e,Hepatobiliary Cancer,Hepatocellular Carcinoma,Liver,Hepatocellular Carcinoma
3,/Users/couci/UTokyo/Research/SyntheticCancerCl...,001944e5-af34-4061-9c09-bb9ea346f6fd,/Users/couci/UTokyo/Research/SyntheticCancerCl...,/Users/couci/UTokyo/Research/SyntheticCancerCl...,TCGA-BLCA,001944e5-af34-4061-9c09-bb9ea346f6fd,Bladder Cancer,Bladder Urothelial Carcinoma,Bladder,Muscle Invasive Urothelial Carcinoma (PT2 or A...
4,/Users/couci/UTokyo/Research/SyntheticCancerCl...,001ad307-4ad3-4f1d-b2fc-efc032871c7e,/Users/couci/UTokyo/Research/SyntheticCancerCl...,/Users/couci/UTokyo/Research/SyntheticCancerCl...,TCGA-LGG,001ad307-4ad3-4f1d-b2fc-efc032871c7e,Glioma,Oligoastrocytoma,CNS,Oligoastrocytoma


In [6]:
df.head()

,RNA-path,id,meth-path,mirna-path,project
0,/Users/couci/UTokyo/Research/SyntheticCancerCl...,001944e5-af34-4061-9c09-bb9ea346f6fd,/Users/couci/UTokyo/Research/SyntheticCancerCl...,/Users/couci/UTokyo/Research/SyntheticCancerCl...,TCGA-BLCA
1,/Users/couci/UTokyo/Research/SyntheticCancerCl...,00d8e96f-f231-4475-9e9f-c1f8aec28e4e,/Users/couci/UTokyo/Research/SyntheticCancerCl...,/Users/couci/UTokyo/Research/SyntheticCancerCl...,TCGA-BLCA
2,/Users/couci/UTokyo/Research/SyntheticCancerCl...,0198d8e5-a14b-44ee-b1c4-d248994845ad,/Users/couci/UTokyo/Research/SyntheticCancerCl...,/Users/couci/UTokyo/Research/SyntheticCancerCl...,TCGA-BLCA
3,NaN,01c815ba-7bda-4f7e-865c-0c5776febf2c,/Users/couci/UTokyo/Research/SyntheticCancerCl...,/Users/couci/UTokyo/Research/SyntheticCancerCl...,TCGA-BLCA
4,/Users/couci/UTokyo/Research/SyntheticCancerCl...,01ee41ef-1894-4408-9e96-a22a1f068190,/Users/couci/UTokyo/Research/SyntheticCancerCl...,/Users/couci/UTokyo/Research/SyntheticCancerCl...,TCGA-BLCA


In [7]:
annotations.head()

,OTHER_PATIENT_ID,CANCER_TYPE,CANCER_TYPE_DETAILED,TUMOR_TISSUE_SITE,TUMOR_TYPE
0,b3164f7b-c826-4e08-9ee6-8ff96d29b913,Adrenocortical Carcinoma,Adrenocortical Carcinoma,Adrenal Gland,"Adrenocortical Carcinoma, Usual Type"
1,8e7c2e31-d085-4b75-a970-162526dd07a0,Adrenocortical Carcinoma,Adrenocortical Carcinoma,Adrenal Gland,"Adrenocortical Carcinoma, Usual Type"
2,dfd687bc-6e69-42f7-af94-d17fc150d1a1,Adrenocortical Carcinoma,Adrenocortical Carcinoma,Adrenal Gland,"Adrenocortical Carcinoma, Usual Type"
3,5f3e2974-f1df-47a2-8a8a-29bb525eeef6,Adrenocortical Carcinoma,Adrenocortical Carcinoma,Adrenal Gland,"Adrenocortical Carcinoma, Usual Type"
4,802dbd0d-ef07-4c91-ab8d-1dd39532e947,Adrenocortical Carcinoma,Adrenocortical Carcinoma,Adrenal Gland,"Adrenocortical Carcinoma, Usual Type"


In [8]:
merged_df.loc[(merged_df["project"] == "normal"), "TUMOR_TYPE"] = "normal"

print(len(df))
print(len(merged_df))
print(len(annotations))

11579
10770
10077


In [9]:
merged_df.head()

,RNA-path,id,meth-path,mirna-path,project,OTHER_PATIENT_ID,CANCER_TYPE,CANCER_TYPE_DETAILED,TUMOR_TISSUE_SITE,TUMOR_TYPE
0,/Users/couci/UTokyo/Research/SyntheticCancerCl...,0004d251-3f70-4395-b175-c94c2f5b1b81,/Users/couci/UTokyo/Research/SyntheticCancerCl...,/Users/couci/UTokyo/Research/SyntheticCancerCl...,TCGA-LIHC,0004d251-3f70-4395-b175-c94c2f5b1b81,Hepatobiliary Cancer,Hepatocellular Carcinoma,Liver,Hepatocellular Carcinoma
1,/Users/couci/UTokyo/Research/SyntheticCancerCl...,000d566c-96c7-4f1c-b36e-fa2222467983,/Users/couci/UTokyo/Research/SyntheticCancerCl...,/Users/couci/UTokyo/Research/SyntheticCancerCl...,TCGA-PRAD,000d566c-96c7-4f1c-b36e-fa2222467983,Prostate Cancer,Prostate Adenocarcinoma,Prostate,"Prostate Adenocarcinoma, Acinar Type"
2,/Users/couci/UTokyo/Research/SyntheticCancerCl...,001887aa-36d0-463f-8bca-dec7043b4f2e,/Users/couci/UTokyo/Research/SyntheticCancerCl...,/Users/couci/UTokyo/Research/SyntheticCancerCl...,TCGA-LIHC,001887aa-36d0-463f-8bca-dec7043b4f2e,Hepatobiliary Cancer,Hepatocellular Carcinoma,Liver,Hepatocellular Carcinoma
3,/Users/couci/UTokyo/Research/SyntheticCancerCl...,001944e5-af34-4061-9c09-bb9ea346f6fd,/Users/couci/UTokyo/Research/SyntheticCancerCl...,/Users/couci/UTokyo/Research/SyntheticCancerCl...,TCGA-BLCA,001944e5-af34-4061-9c09-bb9ea346f6fd,Bladder Cancer,Bladder Urothelial Carcinoma,Bladder,Muscle Invasive Urothelial Carcinoma (PT2 or A...
4,/Users/couci/UTokyo/Research/SyntheticCancerCl...,001ad307-4ad3-4f1d-b2fc-efc032871c7e,/Users/couci/UTokyo/Research/SyntheticCancerCl...,/Users/couci/UTokyo/Research/SyntheticCancerCl...,TCGA-LGG,001ad307-4ad3-4f1d-b2fc-efc032871c7e,Glioma,Oligoastrocytoma,CNS,Oligoastrocytoma


#### Get Images


In [ ]:
import requests, json, time
import pandas as pd

API = "https://api.gdc.cancer.gov/files"

case_ids = merged_df["id"].dropna().unique().tolist()
print(len(case_ids))


def fetch_files_for_batch(batch_case_ids, max_retries=5, backoff=3):
    filters = {
        "op": "and",
        "content": [
            {
                "op": "in",
                "content": {"field": "cases.case_id", "value": batch_case_ids},
            },
            {
                "op": "in",
                "content": {
                    "field": "data_type",
                    "value": ["Slide Image", "Tissue Slide Image"],
                },
            },
            {
                "op": "in",
                "content": {
                    "field": "experimental_strategy",
                    "value": ["Diagnostic Slide"],
                },
            },
        ],
    }

    params = {
        "filters": json.dumps(filters),
        "format": "JSON",
        "fields": "file_id,file_name,cases.case_id",
        "size": "10000",
    }

    for attempt in range(max_retries):
        try:
            r = requests.get(API, params=params, timeout=60)
            r.raise_for_status()
            return r.json()["data"]["hits"]
        except requests.exceptions.RequestException as e:
            print(f"Batch failed (attempt {attempt+1}/{max_retries}): {e}")
            if attempt == max_retries - 1:
                raise
            time.sleep(backoff * (attempt + 1))


all_hits = []
batch_size = 100  # keep small to avoid timeouts / throttling
for i in range(0, len(case_ids), batch_size):
    batch = case_ids[i : i + batch_size]
    print(f"Fetching batch {i//batch_size+1} with {len(batch)} cases")
    hits = fetch_files_for_batch(batch)
    all_hits.extend(hits)

files_df = pd.DataFrame(all_hits)
print("Total slide files found:", len(files_df))

9712
Fetching batch 1 with 100 cases
Fetching batch 2 with 100 cases
Fetching batch 3 with 100 cases
Fetching batch 4 with 100 cases
Fetching batch 5 with 100 cases
Fetching batch 6 with 100 cases
Fetching batch 7 with 100 cases
Fetching batch 8 with 100 cases
Fetching batch 9 with 100 cases
Fetching batch 10 with 100 cases
Fetching batch 11 with 100 cases
Fetching batch 12 with 100 cases
Fetching batch 13 with 100 cases
Fetching batch 14 with 100 cases
Fetching batch 15 with 100 cases
Fetching batch 16 with 100 cases
Fetching batch 17 with 100 cases
Fetching batch 18 with 100 cases
Fetching batch 19 with 100 cases
Fetching batch 20 with 100 cases
Fetching batch 21 with 100 cases
Fetching batch 22 with 100 cases
Fetching batch 23 with 100 cases
Fetching batch 24 with 100 cases
Fetching batch 25 with 100 cases
Fetching batch 26 with 100 cases
Fetching batch 27 with 100 cases
Fetching batch 28 with 100 cases
Fetching batch 29 with 100 cases
Fetching batch 30 with 100 cases
Fetching batch

In [12]:
# save to pickle
files_df.to_pickle("slide_files_df.pkl")
print("Saved slide files to slide_files_df.pkl")

Saved slide files to slide_files_df.pkl


In [13]:
# Save merged_df for use in other notebooks
merged_df.to_pickle("merged_df.pkl")

print("merged_df saved to merged_df.pkl")

merged_df saved to merged_df.pkl


In [4]:
files_df = pd.read_pickle("slide_files_df.pkl")
print("Loaded slide files from slide_files_df.pkl, total files:", len(files_df))
# Create manifest from files_df
with open("gdc_slide_manifest.txt", "w") as f:
    f.write("id\n")
    for file_id in files_df["file_id"]:
        f.write(f"{file_id}\n")
print("Saved manifest to gdc_slide_manifest.txt")

Loaded slide files from slide_files_df.pkl, total files: 10163
Saved manifest to gdc_slide_manifest.txt


In [4]:
files_df.head()

,id,cases,file_name,file_id
0,cea82b7d-135a-49d5-b4f6-3fb0215f7188,[{'case_id': '0045349c-69d9-4306-a403-c9c1fa83...,TCGA-A1-A0SB-01Z-00-DX1.B34C267B-CAAA-4AB6-AD5...,cea82b7d-135a-49d5-b4f6-3fb0215f7188
1,61d03f55-56d6-475f-b390-090320d0ed2d,[{'case_id': '016caf42-4e19-4444-ab5d-6cf1e76c...,TCGA-AO-A128-01Z-00-DX1.4E6BFFBC-87AD-4ED4-959...,61d03f55-56d6-475f-b390-090320d0ed2d
2,363e72db-7642-4be2-b370-27a8d2a104c5,[{'case_id': '001cef41-ff86-4d3f-a140-a647ac4b...,TCGA-E2-A1IU-01Z-00-DX1.E2F24814-24BA-4158-884...,363e72db-7642-4be2-b370-27a8d2a104c5
3,dab8b8a4-84ca-4d1c-873d-66007379f075,[{'case_id': '011b9b2d-ebe5-42bf-9662-d922facc...,TCGA-A7-A26E-01Z-00-DX1.BA4A7E28-0563-4C23-82D...,dab8b8a4-84ca-4d1c-873d-66007379f075
4,11faafb6-b10a-49a5-a55d-d6b9ec843ae8,[{'case_id': '01eef340-598c-4205-a990-cec190ac...,TCGA-B6-A0IQ-01Z-00-DX1.662EA039-825E-41FF-91D...,11faafb6-b10a-49a5-a55d-d6b9ec843ae8


#### Download from gdc-client

```bash
gdc-client download -m gdc_slide_manifest.txt -d Datasets/slides
```

#### Look at svs_preparation for svs cleanup and preparation pipeline

It also includes a cell to download and process the slides 1 by 1 to save as much space as possible


In [5]:
# Check median size of RNA, DNA (methylation), and miRNA parquet files
import os
import pyarrow.parquet as pq

# Load the merged dataframe if not already loaded
try:
    merged_df
except NameError:
    merged_df = pd.read_pickle("merged_df.pkl")
    print("Loaded merged_df from pickle")

# Function to get parquet file sizes
def get_parquet_stats(file_paths, data_type):
    sizes_bytes = []
    num_rows = []
    num_columns = []
    
    for path in file_paths:
        if pd.notna(path) and os.path.exists(path):
            try:
                # Get file size in MB
                file_size_mb = os.path.getsize(path) / (1024 * 1024)
                sizes_bytes.append(file_size_mb)
                
                # Get parquet table info
                parquet_file = pq.read_table(path)
                num_rows.append(parquet_file.num_rows)
                num_columns.append(parquet_file.num_columns)
            except Exception as e:
                print(f"Error reading {path}: {e}")
                continue
    
    if len(sizes_bytes) > 0:
        print(f"\n{data_type} Statistics:")
        print(f"  Total files: {len(sizes_bytes)}")
        print(f"  Median file size: {np.median(sizes_bytes):.2f} MB")
        print(f"  Mean file size: {np.mean(sizes_bytes):.2f} MB")
        print(f"  Median rows: {np.median(num_rows):.0f}")
        print(f"  Median columns: {np.median(num_columns):.0f}")
        print(f"  Min/Max rows: {np.min(num_rows):.0f} / {np.max(num_rows):.0f}")
    else:
        print(f"\n{data_type}: No valid files found")

# Check each data type
print("Analyzing parquet files...")
get_parquet_stats(merged_df['RNA-path'].dropna(), "RNA")
get_parquet_stats(merged_df['meth-path'].dropna(), "DNA (Methylation)")
get_parquet_stats(merged_df['mirna-path'].dropna(), "miRNA")

Loaded merged_df from pickle
Analyzing parquet files...

RNA Statistics:
  Total files: 10204
  Median file size: 1.85 MB
  Mean file size: 1.85 MB
  Median rows: 60660
  Median columns: 9
  Min/Max rows: 60660 / 60660

DNA (Methylation) Statistics:
  Total files: 8955
  Median file size: 6.24 MB
  Mean file size: 6.22 MB
  Median rows: 486427
  Median columns: 2
  Min/Max rows: 486427 / 486427

miRNA Statistics:
  Total files: 10030
  Median file size: 0.02 MB
  Mean file size: 0.02 MB
  Median rows: 1881
  Median columns: 4
  Min/Max rows: 1881 / 1881
